# Avenue Video Anomaly Detection: Hiera-L Feature Extraction Pipeline

This notebook implements a complete, robust, and end-to-end pipeline to extract spatiotemporal features from the **Avenue** dataset using **Hiera-L (Large)**.

### Pipeline Overview
1. **Google Drive Integration**: Mounts Drive and sets up structured path constants.
2. **Avenue Archive Extraction**: Extracts `Avenue_Dataset.zip` from Google Drive into the Colab local filesystem.
3. **Environment & Pinned Dependencies**: Installs `timm`, Hiera (at pinned commit), `decord`, and other necessary libraries.
4. **Dataset Scanning**: Discovers and validates `.avi` files for both the training and testing splits, plus optional `.npy` frame-label files from `ground_truth_avenue`.
5. **Feature Extraction**: Extracts one 1152-D Hiera-L feature vector per video frame using the same centered 16-frame / stride-4 sampling strategy as the ShanghaiTech notebook.
6. **Resume Support & Manifests**: Skips valid existing `.npz` feature files and writes train/test manifests.
7. **Training Statistics & Config**: Computes MULDE-ready train mean/std and writes a consolidated extraction config.

All generated Avenue features are saved separately under:

`/content/drive/MyDrive/MULDE/features/avenue_features`


## Step 1: Mount Google Drive & Configure Paths

We mount Google Drive to `/content/drive` and configure our workspace paths under `/content/drive/MyDrive/MULDE`. The Avenue archive is expected at `/content/drive/MyDrive/Avenue_Dataset.zip`, and the output features are intentionally isolated in `features/avenue_features` to avoid collisions with ShanghaiTech features.


In [1]:
import os
import random
import numpy as np
import torch

# 1. Mount Google Drive
try:
    from google.colab import drive
    print("Colab environment detected. Mounting Google Drive...")
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Google Colab. Skipping Google Drive mount.")

# 2. Define Workspace Paths
# The path constants can be edited at the top of the notebook:
DRIVE_ROOT = "/content/drive/MyDrive/MULDE"

# Avenue raw inputs
AVENUE_ZIP_PATH = "/content/drive/MyDrive/Avenue_Dataset.zip"
AVENUE_EXTRACT_ROOT = "/content/avenue_dataset"
FORCE_REEXTRACT = False

# Optional label folder. If this path does not exist, Step 3 also searches common locations.
GROUND_TRUTH_DIR = "/content/drive/MyDrive/ground_truth_avenue"

# Leave these as None to auto-discover them after unzipping Avenue_Dataset.zip.
# If your zip extracts differently, set them manually to the folders containing .avi files.
TRAIN_VIDEO_DIR = "/content/Avenue Dataset/training_videos"
TEST_VIDEO_DIR = "/content/Avenue Dataset/testing_videos"

# Extracted outputs: separate Avenue folder to prevent overlap/collision with ShanghaiTech features.
FEATURE_DIR = os.path.join(DRIVE_ROOT, "features", "avenue_features")
TRAIN_FEATURE_DIR = os.path.join(FEATURE_DIR, "train")
TEST_FEATURE_DIR = os.path.join(FEATURE_DIR, "test")

# Indexing files
MANIFEST_TRAIN = os.path.join(FEATURE_DIR, "manifest_train.csv")
MANIFEST_TEST = os.path.join(FEATURE_DIR, "manifest_test.csv")
FEATURE_STATS_PATH = os.path.join(FEATURE_DIR, "train_feature_stats.npz")
CONFIG_PATH = os.path.join(FEATURE_DIR, "extraction_config.json")

# Create output directories if they don't exist
os.makedirs(TRAIN_FEATURE_DIR, exist_ok=True)
os.makedirs(TEST_FEATURE_DIR, exist_ok=True)

# 3. Model & Feature Configuration Constants
HIERA_COMMIT = "b12b842542ee5c757fcfec8c41f6b56fcbe89b65"
MODEL_NAME = "hiera_large_16x224"
CHECKPOINT_NAME = "mae_k400_ft_k400"
NUM_FRAMES_PER_CLIP = 16
TEMPORAL_STRIDE = 4
IMAGE_SIZE = 224

# 4. Reproducibility & Random Seeds
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    print(f"Random seed set to: {seed}")

seed_everything(42)

print("\n--- Workspace Paths Configuration ---")
print(f"Avenue Zip: {AVENUE_ZIP_PATH}")
print(f"Extraction Root: {AVENUE_EXTRACT_ROOT}")
print(f"Ground Truth Directory: {GROUND_TRUTH_DIR}")
print(f"Feature Storage: {FEATURE_DIR}")


Colab environment detected. Mounting Google Drive...
Mounted at /content/drive
Random seed set to: 42

--- Workspace Paths Configuration ---
Avenue Zip: /content/drive/MyDrive/Avenue_Dataset.zip
Extraction Root: /content/avenue_dataset
Ground Truth Directory: /content/drive/MyDrive/ground_truth_avenue
Feature Storage: /content/drive/MyDrive/MULDE/features/avenue_features


In [2]:
import os
import glob
import zipfile
import shutil


def directory_has_content(path):
    if not os.path.isdir(path):
        return False
    with os.scandir(path) as entries:
        return any(entries)


def find_split_dir_with_avi(root, keywords):
    """Find the most likely split directory containing Avenue .avi files."""
    if not os.path.isdir(root):
        return None

    candidates = []
    for dirpath, _, filenames in os.walk(root):
        avi_count = sum(1 for name in filenames if name.lower().endswith(".avi"))
        if avi_count == 0:
            continue

        path_text = dirpath.replace("\\", "/").lower()
        base_text = os.path.basename(dirpath).lower()
        if any(keyword in path_text or keyword in base_text for keyword in keywords):
            candidates.append((avi_count, len(dirpath), dirpath))

    if not candidates:
        return None

    # Prefer the directory with the most AVIs, then the shortest path.
    candidates.sort(key=lambda item: (-item[0], item[1]))
    return candidates[0][2]


def find_ground_truth_dir(root, preferred_dir):
    candidates = [
        preferred_dir,
        os.path.join("/content/drive/MyDrive", "ground_truth_avenue"),
        os.path.join(DRIVE_ROOT, "ground_truth_avenue"),
        os.path.join(root, "ground_truth_avenue"),
    ]

    for candidate in candidates:
        if candidate and os.path.isdir(candidate) and glob.glob(os.path.join(candidate, "*.npy")):
            return candidate

    if os.path.isdir(root):
        for dirpath, _, filenames in os.walk(root):
            has_npy = any(name.lower().endswith(".npy") for name in filenames)
            path_text = dirpath.replace("\\", "/").lower()
            looks_like_gt = any(token in path_text for token in ["ground", "truth", "label", "gt"])
            if has_npy and looks_like_gt:
                return dirpath

    return preferred_dir


if FORCE_REEXTRACT and os.path.isdir(AVENUE_EXTRACT_ROOT):
    print(f"FORCE_REEXTRACT=True. Removing existing extraction root: {AVENUE_EXTRACT_ROOT}")
    shutil.rmtree(AVENUE_EXTRACT_ROOT)

if directory_has_content(AVENUE_EXTRACT_ROOT):
    print(f"Avenue archive already extracted at: {AVENUE_EXTRACT_ROOT}")
else:
    if not os.path.exists(AVENUE_ZIP_PATH):
        raise FileNotFoundError(
            f"Avenue zip not found at {AVENUE_ZIP_PATH}. "
            "Upload Avenue_Dataset.zip to MyDrive or edit AVENUE_ZIP_PATH."
        )

    os.makedirs(AVENUE_EXTRACT_ROOT, exist_ok=True)
    print(f"Extracting {AVENUE_ZIP_PATH} to {AVENUE_EXTRACT_ROOT}...")
    with zipfile.ZipFile(AVENUE_ZIP_PATH, "r") as archive:
        archive.extractall(AVENUE_EXTRACT_ROOT)
    print("Extraction complete.")

# Auto-discover split directories if not manually configured.
if TRAIN_VIDEO_DIR is None or not os.path.isdir(TRAIN_VIDEO_DIR):
    TRAIN_VIDEO_DIR = find_split_dir_with_avi(AVENUE_EXTRACT_ROOT, ["train", "training"])

if TEST_VIDEO_DIR is None or not os.path.isdir(TEST_VIDEO_DIR):
    TEST_VIDEO_DIR = find_split_dir_with_avi(AVENUE_EXTRACT_ROOT, ["test", "testing"])

GROUND_TRUTH_DIR = find_ground_truth_dir(AVENUE_EXTRACT_ROOT, GROUND_TRUTH_DIR)

print("\n--- Resolved Avenue Dataset Paths ---")
print(f"Train AVI Directory: {TRAIN_VIDEO_DIR}")
print(f"Test AVI Directory: {TEST_VIDEO_DIR}")
print(f"Ground Truth Directory: {GROUND_TRUTH_DIR}")


Extracting /content/drive/MyDrive/Avenue_Dataset.zip to /content/avenue_dataset...
Extraction complete.

--- Resolved Avenue Dataset Paths ---
Train AVI Directory: /content/avenue_dataset/Avenue Dataset/training_videos
Test AVI Directory: /content/avenue_dataset/Avenue Dataset/testing_videos
Ground Truth Directory: /content/drive/MyDrive/ground_truth_avenue


## Step 2: Install Pinned Dependencies

We install required external libraries including a PyTorch-compatible `timm`, the official Hiera implementation pinned to the exact recommended commit, `decord` for fast hardware-accelerated video decoding, `opencv-python-headless`, `pandas`, and `tqdm`.

In [3]:
# Install pinned dependencies
# Using -q flag to keep the output clean and professional
print("Installing pip dependencies. This might take a minute...")
!pip install -q timm decord opencv-python-headless pandas tqdm
!pip install -q git+https://github.com/facebookresearch/hiera.git@b12b842542ee5c757fcfec8c41f6b56fcbe89b65

print("\nAll dependencies installed successfully:")
import timm
import hiera
import decord
import cv2
import pandas as pd
import tqdm
print(f" - timm: {timm.__version__}")
print(f" - hiera: installed from commit {HIERA_COMMIT[:8]}")
print(f" - decord: {decord.__version__}")
print(f" - cv2 (OpenCV): {cv2.__version__}")
print(f" - pandas: {pd.__version__}")
print(f" - tqdm: {tqdm.__version__}")

Installing pip dependencies. This might take a minute...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 102.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done

All dependencies installed successfully:


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


 - timm: 1.0.27
 - hiera: installed from commit b12b8425
 - decord: 0.6.0
 - cv2 (OpenCV): 4.13.0
 - pandas: 2.2.2
 - tqdm: 4.67.3


## Step 3: Scan Dataset & Validate Structure

We scan the Avenue dataset directory structure to ensure it matches the expected layout. We assert that:
- Training raw files are `.avi` videos.
- Testing raw files are `.avi` videos.
- If `ground_truth_avenue` is available, `.npy` frame-label files can be matched to testing videos where possible.

This step checks data integrity and outputs video/label counts.


In [4]:
import glob
import os
import re


def natural_keys(text):
    return [int(c) if c.isdigit() else c.lower() for c in re.split(r'(\d+)', text)]


def normalize_video_key(value):
    stem = os.path.splitext(os.path.basename(str(value)))[0].lower()
    compact = re.sub(r'[^a-z0-9]+', '', stem)
    digits = re.findall(r'\d+', stem)
    if digits:
        numeric_tail = str(int(digits[-1]))
        return compact, numeric_tail
    return compact, None


def build_label_map(label_files):
    label_map = {}
    for label_path in label_files:
        stem = os.path.splitext(os.path.basename(label_path))[0]
        compact, numeric_tail = normalize_video_key(stem)
        for key in [stem.lower(), compact, numeric_tail]:
            if key:
                label_map.setdefault(key, label_path)
    return label_map


def get_label_path_for_video(video_id):
    if not label_by_video_id:
        return ""

    compact, numeric_tail = normalize_video_key(video_id)
    candidate_keys = [str(video_id).lower(), compact, numeric_tail]
    for key in candidate_keys:
        if key and key in label_by_video_id:
            return label_by_video_id[key]

    # Last-resort suffix matching for names such as video_01.avi vs 01.npy.
    for key, label_path in label_by_video_id.items():
        if compact and key and (compact.endswith(key) or key.endswith(compact)):
            return label_path

    return ""


def scan_avenue_dataset():
    print("Scanning Avenue dataset directories...")

    # 1. Check if directories exist
    if not TRAIN_VIDEO_DIR or not os.path.exists(TRAIN_VIDEO_DIR):
        print(f"Warning: Training directory not found: {TRAIN_VIDEO_DIR}")
        train_videos = []
    else:
        train_videos = sorted(glob.glob(os.path.join(TRAIN_VIDEO_DIR, "*.avi")), key=natural_keys)

    if not TEST_VIDEO_DIR or not os.path.exists(TEST_VIDEO_DIR):
        print(f"Warning: Testing directory not found: {TEST_VIDEO_DIR}")
        test_videos = []
    else:
        test_videos = sorted(glob.glob(os.path.join(TEST_VIDEO_DIR, "*.avi")), key=natural_keys)

    if GROUND_TRUTH_DIR and os.path.exists(GROUND_TRUTH_DIR):
        label_files = sorted(glob.glob(os.path.join(GROUND_TRUTH_DIR, "*.npy")), key=natural_keys)
    else:
        label_files = []
        print(f"Warning: Ground-truth label directory not found: {GROUND_TRUTH_DIR}")

    print(f"Discovered training videos (.avi): {len(train_videos)}")
    print(f"Discovered testing videos (.avi): {len(test_videos)}")
    print(f"Discovered test label files (.npy): {len(label_files)}")

    # 2. Perform validation assertions if any files were found
    if len(train_videos) > 0:
        for path in train_videos:
            assert path.lower().endswith(".avi"), f"Validation Error: Train path {path} must be a .avi file."
        print("✓ All training video paths end with .avi")

    if len(test_videos) > 0:
        for path in test_videos:
            assert path.lower().endswith(".avi"), f"Validation Error: Test path {path} must be a .avi file."
        print("✓ All testing video paths end with .avi")

    global label_by_video_id
    label_by_video_id = build_label_map(label_files)

    if len(test_videos) > 0 and len(label_files) > 0:
        matched = 0
        missing = []
        for video_path in test_videos:
            video_id = os.path.splitext(os.path.basename(video_path))[0]
            if get_label_path_for_video(video_id):
                matched += 1
            else:
                missing.append(video_id)

        print(f"Matched test videos to label files: {matched}/{len(test_videos)}")
        if missing:
            print(f"Warning: No label file matched for first missing videos: {missing[:10]}")

    return train_videos, test_videos


label_by_video_id = {}
train_videos, test_videos = scan_avenue_dataset()


Scanning Avenue dataset directories...
Discovered training videos (.avi): 16
Discovered testing videos (.avi): 21
Discovered test label files (.npy): 21
✓ All training video paths end with .avi
✓ All testing video paths end with .avi
Matched test videos to label files: 21/21


## Step 4: Define Reusable Loading, Preprocessing & Saving Helpers

We implement the core pipeline helpers:
1. **Natural Sort**: Sorts video names numerically (`1.avi`, `2.avi`, etc.) rather than lexicographically.
2. **Video Frame Decoders**: Fast `decord`-based video loading for both training and testing AVI files, with an OpenCV fallback if a video codec is not handled by `decord`.
3. **Clip Index Generator**: Given a target frame `i` and total frames $N$, samples 16 frames with stride 4, centered around `i`:
   $$\text{clip\_indices}[k] = \text{clamp}(i - 30 + 4k, 0, N - 1) \quad \text{for } k=0,\dots,15$$
4. **Preprocessor**: Scales images to $224 \times 224$, converts to RGB, and normalizes using standard Hiera video normalization statistics (`mean=[0.45, 0.45, 0.45]`, `std=[0.225, 0.225, 0.225]`).


In [5]:
import re
import os
import glob
import tempfile
import shutil
import numpy as np
import torch
import cv2
from decord import VideoReader, cpu

# 1. Natural sorting helper
def natural_keys(text):
    return [int(c) if c.isdigit() else c.lower() for c in re.split(r'(\d+)', text)]

# 2. Decode/load AVI video with decord first and OpenCV fallback
def load_video(path):
    try:
        vr = VideoReader(path, ctx=cpu(0))
        num_frames = len(vr)
        fps = float(vr.get_avg_fps())
        if not np.isfinite(fps) or fps <= 0:
            fps = 25.0
        return vr, num_frames, fps
    except Exception as decord_error:
        print(f"[WARNING] decord failed for {path}: {decord_error}")
        print("Falling back to OpenCV VideoCapture for this video.")
        cap = cv2.VideoCapture(path)
        if not cap.isOpened():
            raise ValueError(f"Failed to open video with decord and OpenCV: {path}") from decord_error
        num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        cap.release()
        if not np.isfinite(fps) or fps <= 0:
            fps = 25.0
        return {"backend": "opencv", "path": path}, num_frames, fps

# 3. Split-specific loaders kept for parity with the ShanghaiTech notebook
def load_train_video(path):
    return load_video(path)


def load_test_video(path):
    return load_video(path)

# 4. Generate clip indices with temporal stride 4 centered around target frame i
def generate_clip_indices(i, num_frames):
    indices = []
    for k in range(16):
        idx = i - 30 + 4 * k
        idx = max(0, min(idx, num_frames - 1))
        indices.append(idx)
    return np.array(indices, dtype=np.int64)

# 5. Preprocess a single 16-frame spatiotemporal clip
def preprocess_clip(frames_or_paths, is_train, vr=None, target_size=(224, 224)):
    """
    frames_or_paths: For Avenue videos, this is a list of frame indices loaded via 'vr'.
                    A JPG-path fallback is retained only for compatibility/debugging.
    is_train: bool retained for API parity with the ShanghaiTech notebook.
    vr: VideoReader instance or OpenCV fallback descriptor.
    """
    processed_frames = []

    if vr is not None and hasattr(vr, "get_batch"):
        frames_np = vr.get_batch(frames_or_paths).asnumpy() # [16, H, W, C] RGB
        for img in frames_np:
            img_resized = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
            img_float = img_resized.astype(np.float32) / 255.0
            processed_frames.append(img_float)
    elif vr is not None and isinstance(vr, dict) and vr.get("backend") == "opencv":
        cap = cv2.VideoCapture(vr["path"])
        try:
            for idx in frames_or_paths:
                cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
                ok, img_bgr = cap.read()
                if not ok:
                    raise ValueError(f"Failed to read frame {idx} from {vr['path']}")
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                img_resized = cv2.resize(img_rgb, target_size, interpolation=cv2.INTER_LINEAR)
                img_float = img_resized.astype(np.float32) / 255.0
                processed_frames.append(img_float)
        finally:
            cap.release()
    else:
        # JPG-path fallback retained for compatibility with older experiments.
        for path in frames_or_paths:
            img = cv2.imread(path)
            if img is None:
                raise ValueError(f"Failed to read image at {path}")
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_resized = cv2.resize(img_rgb, target_size, interpolation=cv2.INTER_LINEAR)
            img_float = img_resized.astype(np.float32) / 255.0
            processed_frames.append(img_float)

    # Stack along time axis: [16, H, W, C]
    clip_np = np.stack(processed_frames, axis=0)

    # Convert to PyTorch Tensor: [16, H, W, C] -> [C, T, H, W]
    clip_tensor = torch.tensor(clip_np).permute(3, 0, 1, 2) # [3, 16, 224, 224]

    # Standardize with Hiera-L video stats: mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225]
    mean = torch.tensor([0.45, 0.45, 0.45]).view(3, 1, 1, 1)
    std = torch.tensor([0.225, 0.225, 0.225]).view(3, 1, 1, 1)
    clip_tensor = (clip_tensor - mean) / std

    return clip_tensor # [3, 16, 224, 224]

# 6. Atomic save function for NPZ files (FUSE/Google Drive Safe)
def save_npz_atomic(file_path, data_dict):
    dir_name = os.path.dirname(file_path)
    os.makedirs(dir_name, exist_ok=True)

    # Create temp file on the LOCAL Colab SSD filesystem (guarantees fast, robust POSIX writes)
    fd, temp_path = tempfile.mkstemp(suffix=".npz")
    try:
        os.close(fd)
        np.savez_compressed(temp_path, **data_dict)
        # Use shutil.copy2 to copy from local SSD to Google Drive (sequential FUSE-friendly write)
        shutil.copy2(temp_path, file_path)
        os.remove(temp_path)
    except Exception as e:
        if os.path.exists(temp_path):
            os.remove(temp_path)
        raise e


## Step 5: Load Hiera-L Model & Adapt for Feature Extraction

We download the `hiera_large_16x224` model pretrained via MAE and fine-tuned on Kinetics-400 (`mae_k400_ft_k400`) from PyTorch Hub. To extract the raw, pooled 1152-D spatiotemporal representations instead of standard classification classes, we:
1. Re-assign `model.head = torch.nn.Identity()`.
2. Transition the model to evaluation mode (`model.eval()`).
3. Port the weights to the active CUDA device, if available.
4. Perform a shape validation check with a dummy spatiotemporal tensor $[B, C, T, H, W] = [2, 3, 16, 224, 224]$ to guarantee the output is precisely $[B, 1152]$.

In [6]:
def load_and_validate_hiera():
    print("Loading Hiera-L model (hiera_large_16x224, checkpoint=mae_k400_ft_k400) from PyTorch Hub...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using execution device: {device}")

    # Load pretrained model
    model = torch.hub.load(
        "facebookresearch/hiera",
        model="hiera_large_16x224",
        pretrained=True,
        checkpoint="mae_k400_ft_k400"
    )

    # Print original classification head details
    print(f"Original model head: {model.head}")

    # Replace the head with Identity to extract pooled features
    model.head = torch.nn.Identity()
    print("Modified model head to torch.nn.Identity() for feature extraction.")

    model = model.to(device)
    model.eval()

    # Validate with a dummy tensor of shape [B, C, T, H, W]
    dummy_input = torch.randn(2, 3, 16, 224, 224, device=device)
    with torch.no_grad():
        dummy_output = model(dummy_input)

    print(f"Dummy Input shape: {dummy_input.shape}")
    print(f"Dummy Output shape: {dummy_output.shape}")

    assert dummy_output.shape == (2, 1152), f"Validation Error: Expected output shape [2, 1152], got {dummy_output.shape}"
    print("✓ Model output shape validation successful! Hiera-L correctly produces 1152-D spatiotemporal vectors.")

    return model, device

model, device = load_and_validate_hiera()

Loading Hiera-L model (hiera_large_16x224, checkpoint=mae_k400_ft_k400) from PyTorch Hub...
Using execution device: cuda
Downloading: "https://github.com/facebookresearch/hiera/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/hiera/hiera_large_16x224.pth" to /root/.cache/torch/hub/checkpoints/hiera_large_16x224.pth


100%|██████████| 2.38G/2.38G [00:50<00:00, 50.2MB/s]


Original model head: Head(
  (dropout): Identity()
  (projection): Linear(in_features=1152, out_features=400, bias=True)
)
Modified model head to torch.nn.Identity() for feature extraction.
Dummy Input shape: torch.Size([2, 3, 16, 224, 224])
Dummy Output shape: torch.Size([2, 1152])
✓ Model output shape validation successful! Hiera-L correctly produces 1152-D spatiotemporal vectors.


## Step 6: End-to-End Smoke Testing

We run a lightweight smoke test to verify our loading, preprocessing, model execution, indexing, and atomic NPZ-saving pipeline.
Specifically, if raw dataset files exist, we:
1. Process a single training AVI video and save its `.npz` feature file.
2. Process a single testing AVI video and save its `.npz` feature file.
3. Validate frame counts: check that `features.shape[0]` matches the number of frames.
4. Validate feature content: confirm there are no `NaN` or `Inf` values.


In [7]:
import os
import gc
import cv2
import json
import datetime
import tqdm
import shutil
import numpy as np
import torch

# =============================================================================
# Pre-caching functions: decode & preprocess every frame exactly ONCE.
# This eliminates the 16x redundant decoding that caused the 7-minute bottleneck.
# Memory: a 764-frame video cached at [764, 3, 224, 224] float32 ≈ 460 MB.
# =============================================================================

def preprocess_all_frames_from_video(video_reader, num_frames, target_size=(224, 224), chunk_size=128):
    """Pre-decode and preprocess ALL frames from an AVI video.
    Uses chunked decord loading when available to limit peak memory.
    Falls back to sequential OpenCV reads when decord cannot decode the video.
    Returns numpy array [num_frames, 3, 224, 224] (float32, Hiera-normalized).
    """
    mean = np.array([0.45, 0.45, 0.45], dtype=np.float32).reshape(1, 3, 1, 1)
    std  = np.array([0.225, 0.225, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

    all_frames = np.empty((num_frames, 3, target_size[0], target_size[1]), dtype=np.float32)

    if hasattr(video_reader, "get_batch"):
        for start in range(0, num_frames, chunk_size):
            end = min(start + chunk_size, num_frames)
            indices = list(range(start, end))
            frames_np = video_reader.get_batch(indices).asnumpy()  # [chunk, H, W, C] RGB uint8

            for j, img in enumerate(frames_np):
                img_resized = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
                img_float = img_resized.astype(np.float32) / 255.0
                all_frames[start + j] = img_float.transpose(2, 0, 1)  # HWC -> CHW

            del frames_np
    elif isinstance(video_reader, dict) and video_reader.get("backend") == "opencv":
        cap = cv2.VideoCapture(video_reader["path"])
        try:
            for frame_idx in range(num_frames):
                ok, img_bgr = cap.read()
                if not ok:
                    raise ValueError(f"Failed to read frame {frame_idx} from {video_reader['path']}")
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                img_resized = cv2.resize(img_rgb, target_size, interpolation=cv2.INTER_LINEAR)
                img_float = img_resized.astype(np.float32) / 255.0
                all_frames[frame_idx] = img_float.transpose(2, 0, 1)  # HWC -> CHW
        finally:
            cap.release()
    else:
        raise TypeError("Unsupported video reader type.")

    # Vectorized Hiera normalization across all frames at once
    all_frames = (all_frames - mean) / std
    return all_frames  # [num_frames, 3, 224, 224]


def preprocess_all_frames_train(vr, num_frames, target_size=(224, 224), chunk_size=128):
    """Pre-decode and preprocess ALL frames from a training AVI video."""
    return preprocess_all_frames_from_video(vr, num_frames, target_size=target_size, chunk_size=chunk_size)


def preprocess_all_frames_test(vr, num_frames, target_size=(224, 224), chunk_size=128):
    """Pre-decode and preprocess ALL frames from a testing AVI video."""
    return preprocess_all_frames_from_video(vr, num_frames, target_size=target_size, chunk_size=chunk_size)


# =============================================================================
# Main extraction function (optimized with frame pre-caching, dynamic OOM recovery, and FP16 autocast)
# =============================================================================

def extract_features_for_video(video_source, is_train, out_npz_path, model, device, batch_size=8):
    """
    Optimized feature extraction with pre-cached preprocessed frames.
    Each frame is decoded, resized, and normalized exactly ONCE.
    Clips are assembled via fast numpy index slicing (zero re-decoding).

    Includes dynamic CUDA OOM recovery: if the GPU runs out of memory,
    it automatically halves the batch size and retries.

    Uses torch.cuda.amp.autocast to leverage T4 Tensor Cores for massive speedups.
    Returns (out_npz_path, num_frames, fps)
    """
    if video_source == 'mock':
        # Synthetic mock extraction (unchanged)
        num_frames = 45
        fps = 24.0
        video_id = "mock_video_01"
        split = "train" if is_train else "test"
        source_path = "mock_source"

        print(f"Running mock feature extraction for {video_id} ({split})...")
        features = np.random.randn(num_frames, 1152).astype(np.float32)
        frame_indices = np.arange(num_frames, dtype=np.int64)
        clip_indices = np.zeros((num_frames, 16), dtype=np.int64)
        for i in range(num_frames):
            clip_indices[i] = generate_clip_indices(i, num_frames)
    else:
        # Real extraction
        split = "train" if is_train else "test"
        video_id = os.path.splitext(os.path.basename(video_source))[0]
        if is_train:
            vr, num_frames, fps = load_train_video(video_source)
        else:
            vr, num_frames, fps = load_test_video(video_source)

        source_path = video_source
        print(f"Extracting features for {video_id} ({split})... Total frames: {num_frames}")

        # ---- PHASE 1: Pre-cache all preprocessed frames (each frame decoded ONCE) ----
        print(f"  Phase 1/2: Pre-caching {num_frames} preprocessed frames...")
        if is_train:
            cached_frames = preprocess_all_frames_train(vr, num_frames)
        else:
            cached_frames = preprocess_all_frames_test(vr, num_frames)
        del vr  # Release decoder/fallback descriptor to free memory

        gc.collect()
        mem_mb = cached_frames.nbytes / (1024 * 1024)
        print(f"  Frame cache ready: {cached_frames.shape} — {mem_mb:.1f} MB")

        # ---- PHASE 2: Assemble clips via slicing + batched GPU inference (with OOM recovery) ----
        print(f"  Phase 2/2: Running batched Hiera-L inference...")
        frame_indices = np.arange(num_frames, dtype=np.int64)
        clip_indices = np.zeros((num_frames, 16), dtype=np.int64)

        # Pre-compute all clip indices
        for i in range(num_frames):
            clip_indices[i] = generate_clip_indices(i, num_frames)

        current_batch_size = batch_size
        success = False
        features_list = []

        while not success and current_batch_size >= 1:
            try:
                features_list = []
                # Batched inference loop
                for batch_start in tqdm.tqdm(range(0, num_frames, current_batch_size), desc=f"Batches (size={current_batch_size})"):
                    batch_end = min(batch_start + current_batch_size, num_frames)
                    batch_clips = []

                    for i in range(batch_start, batch_end):
                        # Fast numpy slicing from the pre-cached array
                        clip_frames = cached_frames[clip_indices[i]]   # [16, 3, 224, 224]
                        # Permute to Hiera input format: [C, T, H, W]
                        clip_tensor = torch.from_numpy(clip_frames.copy()).permute(1, 0, 2, 3)
                        batch_clips.append(clip_tensor)

                    # Stack to [B, C, T, H, W] and move to GPU
                    stacked = torch.stack(batch_clips, dim=0).to(device)
                    with torch.no_grad():
                        # Enable mixed precision (autocast) on CUDA to leverage T4 Tensor Cores
                        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                            feats = model(stacked)  # [B, 1152]
                        # Ensure features are converted back to FP32 before saving
                        features_list.append(feats.float().cpu().numpy())

                    del stacked, batch_clips

                features = np.concatenate(features_list, axis=0)  # [num_frames, 1152]
                success = True
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"\n[WARNING] CUDA Out of Memory with batch_size={current_batch_size}. Retrying with batch_size={current_batch_size // 2}...")
                    current_batch_size //= 2
                    # Clear variables to recover memory
                    if 'stacked' in locals(): del stacked
                    if 'batch_clips' in locals(): del batch_clips
                    if 'features_list' in locals(): del features_list
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    if current_batch_size < 1:
                        raise e
                else:
                    raise e

        # Free the cache to reclaim RAM
        del cached_frames
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Package metadata as a compact JSON string
    metadata = {
        "dataset": "Avenue",
        "model": MODEL_NAME,
        "checkpoint": CHECKPOINT_NAME,
        "hiera_commit": HIERA_COMMIT,
        "extraction_date": datetime.datetime.now().isoformat(),
        "pytorch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "temporal_stride": TEMPORAL_STRIDE,
        "clip_len": NUM_FRAMES_PER_CLIP,
        "preprocessing": {
            "resize": f"{IMAGE_SIZE}x{IMAGE_SIZE}",
            "mean": [0.45, 0.45, 0.45],
            "std": [0.225, 0.225, 0.225]
        }
    }

    data_to_save = {
        "features": features.astype(np.float32),
        "frame_indices": frame_indices.astype(np.int64),
        "clip_indices": clip_indices.astype(np.int64),
        "video_id": video_id,
        "split": split,
        "source_path": source_path,
        "num_frames": int(num_frames),
        "fps": float(fps),
        "metadata_json": json.dumps(metadata, indent=2)
    }

    save_npz_atomic(out_npz_path, data_to_save)
    print(f"✓ Saved: {out_npz_path}")
    return out_npz_path, int(num_frames), float(fps)

# Execute Smoke Test
print("--- Launching Smoke Test ---")
# Use local fast SSD files for smoke test validation to completely bypass Google Drive FUSE latency
local_smoke_train_out = "/content/smoke_test_train_local.npz"
local_smoke_test_out = "/content/smoke_test_test_local.npz"

smoke_train_out = os.path.join(TRAIN_FEATURE_DIR, "smoke_test_video.npz")
smoke_test_out = os.path.join(TEST_FEATURE_DIR, "smoke_test_video.npz")

if len(train_videos) > 0 and len(test_videos) > 0:
    # Use actual videos
    print("Real Avenue dataset files discovered. Running real smoke test...")
    real_train_video = train_videos[0]
    real_test_video = test_videos[0]
    extract_features_for_video(real_train_video, is_train=True, out_npz_path=local_smoke_train_out, model=model, device=device)
    extract_features_for_video(real_test_video, is_train=False, out_npz_path=local_smoke_test_out, model=model, device=device)
else:
    # Dry run mock smoke test
    print("No raw dataset files detected at paths. Executing mock smoke test...")
    extract_features_for_video("mock", is_train=True, out_npz_path=local_smoke_train_out, model=model, device=device)
    extract_features_for_video("mock", is_train=False, out_npz_path=local_smoke_test_out, model=model, device=device)

# Verify reloading and statistics on LOCAL SSD paths (100% reliable)
for path in [local_smoke_train_out, local_smoke_test_out]:
    assert os.path.exists(path), f"Smoke Test Failure: Output file {path} was not created."
    loaded = np.load(path)
    print(f"\nVerifying loaded file: {os.path.basename(path)}")
    print(f" - features shape: {loaded['features'].shape}")
    print(f" - num_frames: {loaded['num_frames']}")
    print(f" - video_id: {loaded['video_id']}")

    # Check NaN/Inf
    assert np.isfinite(loaded['features']).all(), "Smoke Test Failure: Features contain non-finite values (NaN/Inf)."
    assert loaded['features'].shape[1] == 1152, f"Smoke Test Failure: Expected feature size 1152, got {loaded['features'].shape[1]}"
    print(" - NaN/Inf check: Passed (all values finite)")
    print(" - Feature dimensions check: Passed")

    # Print sample metadata
    meta = json.loads(str(loaded['metadata_json']))
    print(f" - Metadata loaded: model={meta['model']}, date={meta['extraction_date']}")

# After successful verification, copy the smoke test files to their final Google Drive destinations
print("\nCopying verified smoke test artifacts to Google Drive...")
shutil.copy2(local_smoke_train_out, smoke_train_out)
shutil.copy2(local_smoke_test_out, smoke_test_out)
print(f"✓ Saved to Drive: {smoke_train_out}")
print(f"✓ Saved to Drive: {smoke_test_out}")

print("\n✓ Smoke test executed and validated successfully!")


--- Launching Smoke Test ---
Real Avenue dataset files discovered. Running real smoke test...
Extracting features for 01 (train)... Total frames: 1364
  Phase 1/2: Pre-caching 1364 preprocessed frames...
  Frame cache ready: (1364, 3, 224, 224) — 783.2 MB
  Phase 2/2: Running batched Hiera-L inference...


Batches (size=8): 100%|██████████| 171/171 [05:41<00:00,  2.00s/it]


✓ Saved: /content/smoke_test_train_local.npz
Extracting features for 01 (test)... Total frames: 1439
  Phase 1/2: Pre-caching 1439 preprocessed frames...
  Frame cache ready: (1439, 3, 224, 224) — 826.3 MB
  Phase 2/2: Running batched Hiera-L inference...


Batches (size=8): 100%|██████████| 180/180 [06:02<00:00,  2.01s/it]


✓ Saved: /content/smoke_test_test_local.npz

Verifying loaded file: smoke_test_train_local.npz
 - features shape: (1364, 1152)
 - num_frames: 1364
 - video_id: 01
 - NaN/Inf check: Passed (all values finite)
 - Feature dimensions check: Passed
 - Metadata loaded: model=hiera_large_16x224, date=2026-06-29T22:37:13.409515

Verifying loaded file: smoke_test_test_local.npz
 - features shape: (1439, 1152)
 - num_frames: 1439
 - video_id: 01
 - NaN/Inf check: Passed (all values finite)
 - Feature dimensions check: Passed
 - Metadata loaded: model=hiera_large_16x224, date=2026-06-29T22:43:20.122631

Copying verified smoke test artifacts to Google Drive...
✓ Saved to Drive: /content/drive/MyDrive/MULDE/features/avenue_features/train/smoke_test_video.npz
✓ Saved to Drive: /content/drive/MyDrive/MULDE/features/avenue_features/test/smoke_test_video.npz

✓ Smoke test executed and validated successfully!


## Step 7: Full Dataset Feature Extraction (with Resume Support)

We run the main extraction loops over all training and testing AVI videos.
To make the extraction extremely robust against timeouts, Google Drive disconnections, or runtime failures:
1. **Resume Support**: If a valid `.npz` file already exists at the target path, we skip extracting it.
2. **Atomic Writes**: We perform all writes to a temporary file first and then rename/copy it, guaranteeing that no partial or corrupt `.npz` files are stored.
3. **Manifest Compilation**: As we run, we track status and build high-quality structured metadata lists for manifest generation. The test manifest also includes a `label_path` column when matching `.npy` labels are found.



In [8]:
import os
import traceback
import numpy as np
import pandas as pd

def verify_existing_npz(path):
    """
    Returns True if path exists and is a valid non-corrupt npz file with the required keys.
    """
    if not os.path.exists(path):
        return False
    try:
        loaded = np.load(path)
        required_keys = ['features', 'frame_indices', 'clip_indices', 'video_id', 'split', 'num_frames']
        for k in required_keys:
            if k not in loaded:
                return False
        # Verify shape
        if loaded['features'].ndim != 2 or loaded['features'].shape[1] != 1152:
            return False
        return True
    except Exception:
        return False

def run_full_extraction(train_videos, test_videos, model, device, batch_size=16):
    print("--- Initiating Full Feature Extraction Pipeline ---")

    train_manifest_rows = []
    test_manifest_rows = []

    # 1. Process Training Videos
    if len(train_videos) == 0:
        print("No raw training videos found to extract. Skipping train split extraction loop.")
    else:
        print(f"Starting Train split extraction: {len(train_videos)} videos...")
        for i, video_path in enumerate(train_videos):
            video_id = os.path.splitext(os.path.basename(video_path))[0]
            feature_path = os.path.join(TRAIN_FEATURE_DIR, f"{video_id}.npz")

            # Setup initial row
            row = {
                "split": "train",
                "video_id": video_id,
                "source_path": video_path,
                "feature_path": feature_path,
                "label_path": "",
                "num_frames": 0,
                "fps": 0.0,
                "status": "pending"
            }

            try:
                if verify_existing_npz(feature_path):
                    print(f"[{i+1}/{len(train_videos)}] Skipping (valid NPZ already exists): {video_id}")
                    loaded = np.load(feature_path)
                    row["num_frames"] = int(loaded["num_frames"])
                    row["fps"] = float(loaded["fps"])
                    row["status"] = "success"
                else:
                    print(f"[{i+1}/{len(train_videos)}] Extracting features for: {video_id}")
                    # Extract features directly, catching returned frames and fps
                    _, num_frames, fps = extract_features_for_video(video_path, is_train=True, out_npz_path=feature_path, model=model, device=device, batch_size=batch_size)

                    row["num_frames"] = num_frames
                    row["fps"] = fps
                    row["status"] = "success"
            except Exception as e:
                print(f"Error extracting features for video {video_id}: {str(e)}")
                traceback.print_exc()
                row["status"] = f"error: {str(e)}"

            train_manifest_rows.append(row)

    # 2. Process Testing Videos
    if len(test_videos) == 0:
        print("No raw testing videos found to extract. Skipping test split extraction loop.")
    else:
        print(f"\nStarting Test split extraction: {len(test_videos)} videos...")
        for i, video_path in enumerate(test_videos):
            video_id = os.path.splitext(os.path.basename(video_path))[0]
            feature_path = os.path.join(TEST_FEATURE_DIR, f"{video_id}.npz")
            label_path = get_label_path_for_video(video_id) if 'get_label_path_for_video' in globals() else ""

            row = {
                "split": "test",
                "video_id": video_id,
                "source_path": video_path,
                "feature_path": feature_path,
                "label_path": label_path,
                "num_frames": 0,
                "fps": 0.0,
                "status": "pending"
            }

            try:
                if verify_existing_npz(feature_path):
                    print(f"[{i+1}/{len(test_videos)}] Skipping (valid NPZ already exists): {video_id}")
                    loaded = np.load(feature_path)
                    row["num_frames"] = int(loaded["num_frames"])
                    row["fps"] = float(loaded["fps"])
                    row["status"] = "success"
                else:
                    print(f"[{i+1}/{len(test_videos)}] Extracting features for video: {video_id}")
                    _, num_frames, fps = extract_features_for_video(video_path, is_train=False, out_npz_path=feature_path, model=model, device=device, batch_size=batch_size)

                    row["num_frames"] = num_frames
                    row["fps"] = fps
                    row["status"] = "success"
            except Exception as e:
                print(f"Error extracting features for test video {video_id}: {str(e)}")
                traceback.print_exc()
                row["status"] = f"error: {str(e)}"

            test_manifest_rows.append(row)

    # 3. Save manifests as CSV files
    if len(train_manifest_rows) > 0:
        df_train = pd.DataFrame(train_manifest_rows)
        df_train.to_csv(MANIFEST_TRAIN, index=False)
        print(f"\n✓ Saved training manifest to: {MANIFEST_TRAIN}")
    if len(test_manifest_rows) > 0:
        df_test = pd.DataFrame(test_manifest_rows)
        df_test.to_csv(MANIFEST_TEST, index=False)
        print(f"✓ Saved testing manifest to: {MANIFEST_TEST}")

    print("\n--- Feature Extraction Cycle Complete ---")
    return train_manifest_rows, test_manifest_rows

train_rows, test_rows = run_full_extraction(train_videos, test_videos, model, device)


--- Initiating Full Feature Extraction Pipeline ---
Starting Train split extraction: 16 videos...
[1/16] Extracting features for: 01
Extracting features for 01 (train)... Total frames: 1364
  Phase 1/2: Pre-caching 1364 preprocessed frames...
  Frame cache ready: (1364, 3, 224, 224) — 783.2 MB
  Phase 2/2: Running batched Hiera-L inference...


Batches (size=16): 100%|██████████| 86/86 [05:45<00:00,  4.01s/it]


✓ Saved: /content/drive/MyDrive/MULDE/features/avenue_features/train/01.npz
[2/16] Extracting features for: 02
Extracting features for 02 (train)... Total frames: 1511
  Phase 1/2: Pre-caching 1511 preprocessed frames...
  Frame cache ready: (1511, 3, 224, 224) — 867.6 MB
  Phase 2/2: Running batched Hiera-L inference...


Batches (size=16): 100%|██████████| 95/95 [06:21<00:00,  4.02s/it]


✓ Saved: /content/drive/MyDrive/MULDE/features/avenue_features/train/02.npz
[3/16] Extracting features for: 03
Extracting features for 03 (train)... Total frames: 1487
  Phase 1/2: Pre-caching 1487 preprocessed frames...
  Frame cache ready: (1487, 3, 224, 224) — 853.9 MB
  Phase 2/2: Running batched Hiera-L inference...


...

✓ Saved: /content/drive/MyDrive/MULDE/features/avenue_features/test/19.npz
[20/21] Extracting features for video: 20
Extracting features for 20 (test)... Total frames: 273
  Phase 1/2: Pre-caching 273 preprocessed frames...
  Frame cache ready: (273, 3, 224, 224) — 156.8 MB
  Phase 2/2: Running batched Hiera-L inference...


Batches (size=16): 100%|██████████| 18/18 [01:09<00:00,  3.84s/it]


✓ Saved: /content/drive/MyDrive/MULDE/features/avenue_features/test/20.npz
[21/21] Extracting features for video: 21
Extracting features for 21 (test)... Total frames: 76
  Phase 1/2: Pre-caching 76 preprocessed frames...
  Frame cache ready: (76, 3, 224, 224) — 43.6 MB
  Phase 2/2: Running batched Hiera-L inference...


Batches (size=16): 100%|██████████| 5/5 [00:19<00:00,  3.84s/it]


✓ Saved: /content/drive/MyDrive/MULDE/features/avenue_features/test/21.npz

✓ Saved training manifest to: /content/drive/MyDrive/MULDE/features/avenue_features/manifest_train.csv
✓ Saved testing manifest to: /content/drive/MyDrive/MULDE/features/avenue_features/manifest_test.csv

--- Feature Extraction Cycle Complete ---


## Step 8: Compute Training Set Feature Statistics (Mean & Std)

MULDE is trained on standardized video features. Because loading the entire training set (up to 300,000 frames) simultaneously could easily lead to RAM exhaustion, we compute the global training-set mean and standard deviation using a **memory-efficient streaming statistics algorithm**.
We read each `.npz` file from the train folder sequentially, accumulating feature sums and squared sums frame-by-frame, and then compute:
- **Global Mean Vector**: `[1152]`
- **Global Standard Deviation Vector**: `[1152]`
These stats are stored as `train_feature_stats.npz` in the output feature directory, ready to be read by the downstream score-matching model.

In [9]:
import os
import glob
import tqdm
import datetime
import numpy as np

def compute_train_statistics():
    print("--- Computing Global Training Set Feature Statistics ---")

    # 1. Discover all train feature files
    all_train_files = sorted(glob.glob(os.path.join(TRAIN_FEATURE_DIR, "*.npz")))

    # Filter out smoke test file so it doesn't double count or contaminate stats
    all_train_files = [f for f in all_train_files if "smoke_test" not in os.path.basename(f)]

    if len(all_train_files) == 0:
        print("No real training feature files found! Checking for smoke test file...")
        # If there are no other training files, fallback to smoke_test for dry-run continuity
        all_train_files = sorted(glob.glob(os.path.join(TRAIN_FEATURE_DIR, "smoke_test_video.npz")))
        if len(all_train_files) == 0:
            print("No training feature files found! Cannot calculate statistics.")
            return None

    print(f"Streaming through {len(all_train_files)} training feature files...")

    n_frames = 0
    sum_x = np.zeros(1152, dtype=np.float64)
    sum_x2 = np.zeros(1152, dtype=np.float64)

    for path in tqdm.tqdm(all_train_files, desc="Streaming Stats"):
        try:
            data = np.load(path)
            feats = data['features'].astype(np.float64) # [num_frames, 1152]

            sum_x += feats.sum(axis=0)
            sum_x2 += (feats ** 2).sum(axis=0)
            n_frames += feats.shape[0]
        except Exception as e:
            print(f"Skipping file {path} due to load error: {e}")

    if n_frames == 0:
        print("Total loaded training frames is 0. Cannot compute stats.")
        return None

    # Calculate global mean and standard deviation
    global_mean = sum_x / n_frames
    global_variance = (sum_x2 / n_frames) - (global_mean ** 2)
    # Clamp small negative variances from numerical errors to 0
    global_variance = np.clip(global_variance, 0.0, None)
    global_std = np.sqrt(global_variance)

    # Check for features with zero variance to avoid divide-by-zero during normalization
    zero_var_mask = global_std < 1e-8
    if zero_var_mask.any():
        print(f"Warning: Found {zero_var_mask.sum()} feature dimensions with near-zero variance. Fixing std to 1.0 for these features.")
        global_std[zero_var_mask] = 1.0

    print(f"\nSuccessfully accumulated statistics over {n_frames} frames.")
    print(f"Mean Vector - min: {global_mean.min():.5f}, max: {global_mean.max():.5f}, avg: {global_mean.mean():.5f}")
    print(f"Std Vector  - min: {global_std.min():.5f}, max: {global_std.max():.5f}, avg: {global_std.mean():.5f}")

    # Save statistics atomically
    stats_dict = {
        "mean": global_mean.astype(np.float32),
        "std": global_std.astype(np.float32),
        "num_frames": int(n_frames),
        "computed_at": datetime.datetime.now().isoformat()
    }

    save_npz_atomic(FEATURE_STATS_PATH, stats_dict)
    print(f"✓ Saved training feature statistics to: {FEATURE_STATS_PATH}")
    return FEATURE_STATS_PATH

stats_path = compute_train_statistics()

--- Computing Global Training Set Feature Statistics ---
Streaming through 16 training feature files...


Streaming Stats: 100%|██████████| 16/16 [00:00<00:00, 20.82it/s]


Successfully accumulated statistics over 15328 frames.
Mean Vector - min: -0.76445, max: 0.70744, avg: 0.00072
Std Vector  - min: 0.01078, max: 0.21598, avg: 0.10300
✓ Saved training feature statistics to: /content/drive/MyDrive/MULDE/features/avenue_features/train_feature_stats.npz


## Step 9: Generate Consolidated MULDE-Ready Config Index

We compile a unified JSON configuration file containing:
1. **Dataset Specs**: Avenue archive, resolved train/test video directories, and optional ground-truth label directory.
2. **Model Specs**: backbone type, pretrained source, and pinned repository details.
3. **Preprocessing Details**: spatiotemporal frame sampling strides, dimensions, crop strategy, and mean/std normalization vectors.
4. **Statistical Path**: relative location of the global `train_feature_stats.npz` file.
5. **Manifest Paths**: locations of train and test manifests.
6. **Execution Environment**: PyTorch versions, OS split, and completion timestamp.


In [10]:
import os
import json
import datetime
import torch

def generate_mulde_config_index():
    print("--- Creating Consolidated MULDE Config Index ---")

    # Assemble comprehensive metadata index
    config_dict = {
        "dataset": "Avenue",
        "raw_inputs": {
            "avenue_zip_path": AVENUE_ZIP_PATH,
            "avenue_extract_root": AVENUE_EXTRACT_ROOT,
            "train_video_directory": TRAIN_VIDEO_DIR,
            "test_video_directory": TEST_VIDEO_DIR,
            "ground_truth_directory": GROUND_TRUTH_DIR
        },
        "model_name": MODEL_NAME,
        "checkpoint_name": CHECKPOINT_NAME,
        "hiera_github_commit": HIERA_COMMIT,
        "feature_dimension": 1152,
        "spatiotemporal_sampling": {
            "clip_length": NUM_FRAMES_PER_CLIP,
            "temporal_stride": TEMPORAL_STRIDE,
            "alignment": "centered_around_target",
            "frame_selection_formula": "clamp(target - 30 + 4*k, 0, N-1) for k=0..15"
        },
        "preprocessing_pipeline": {
            "resize_dimensions": [IMAGE_SIZE, IMAGE_SIZE],
            "crop_policy": "full_frame_resize",
            "color_space": "RGB",
            "intensity_scaling": "[0.0, 1.0]",
            "hiera_normalization_stats": {
                "mean": [0.45, 0.45, 0.45],
                "std": [0.225, 0.225, 0.225]
            }
        },
        "artifacts_registry": {
            "root_feature_directory": FEATURE_DIR,
            "relative_train_directory": "train",
            "relative_test_directory": "test",
            "train_manifest_csv": os.path.basename(MANIFEST_TRAIN),
            "test_manifest_csv": os.path.basename(MANIFEST_TEST),
            "train_feature_stats_npz": os.path.basename(FEATURE_STATS_PATH)
        },
        "system_metadata": {
            "pytorch_version": torch.__version__,
            "torch_cuda_available": torch.cuda.is_available(),
            "creation_time": datetime.datetime.now().isoformat()
        }
    }

    # Write JSON atomically/safely
    try:
        with open(CONFIG_PATH, "w") as f:
            json.dump(config_dict, f, indent=2)
        print(f"✓ Consolidated configuration successfully written to: {CONFIG_PATH}")

        # Display the configuration to the user
        print("\n--- Consolidated Extraction Configuration JSON ---")
        print(json.dumps(config_dict, indent=2))

    except Exception as e:
        print(f"Error saving consolidated configuration: {e}")

generate_mulde_config_index()


--- Creating Consolidated MULDE Config Index ---
✓ Consolidated configuration successfully written to: /content/drive/MyDrive/MULDE/features/avenue_features/extraction_config.json

--- Consolidated Extraction Configuration JSON ---
{
  "dataset": "Avenue",
  "raw_inputs": {
    "avenue_zip_path": "/content/drive/MyDrive/Avenue_Dataset.zip",
    "avenue_extract_root": "/content/avenue_dataset",
    "train_video_directory": "/content/avenue_dataset/Avenue Dataset/training_videos",
    "test_video_directory": "/content/avenue_dataset/Avenue Dataset/testing_videos",
    "ground_truth_directory": "/content/drive/MyDrive/ground_truth_avenue"
  },
  "model_name": "hiera_large_16x224",
  "checkpoint_name": "mae_k400_ft_k400",
  "hiera_github_commit": "b12b842542ee5c757fcfec8c41f6b56fcbe89b65",
  "feature_dimension": 1152,
  "spatiotemporal_sampling": {
    "clip_length": 16,
    "temporal_stride": 4,
    "alignment": "centered_around_target",
    "frame_selection_formula": "clamp(target - 30 +

## Pipeline Execution & Verification Plan

### Automated Verification
Once this notebook is executed in a Colab session:
1. **Smoke Test Output Verification**:
   - Verify `train/smoke_test_video.npz` and `test/smoke_test_video.npz` are created and contain `features` arrays of size `[num_frames, 1152]`.
   - Confirm that the features contain no `NaN` or `Inf` values and have non-zero variance.
2. **Boundary Clamping Verification**:
   - Confirm that `clip_indices` for the first and last frames stay within valid bounds.
   - Confirm that the centered stride-4 sampling formula is preserved from the ShanghaiTech notebook.
3. **Manifest Verification**:
   - Confirm `manifest_train.csv` and `manifest_test.csv` exist under `/content/drive/MyDrive/MULDE/features/avenue_features`.
   - Confirm every successful row points to an existing `.npz` file.
   - Confirm the test manifest includes matched `label_path` values when `ground_truth_avenue` is available.
4. **Statistics Verification**:
   - Confirm `train_feature_stats.npz` exists and contains `mean`, `std`, and `num_frames`.
   - Confirm `mean.shape == (1152,)` and `std.shape == (1152,)`.
5. **Configuration Verification**:
   - Confirm `extraction_config.json` exists and points to the Avenue feature directory, manifests, stats, raw AVI directories, and ground-truth directory.

### Expected Final Output Layout

```text
/content/drive/MyDrive/MULDE/features/avenue_features/
├── train/
│   ├── <train_video_id>.npz
│   └── ...
├── test/
│   ├── <test_video_id>.npz
│   └── ...
├── manifest_train.csv
├── manifest_test.csv
├── train_feature_stats.npz
└── extraction_config.json
```
